# Separate a song into vocals and instrumental

Point `SONG` at an mp3 and run all cells. You get two mp3 files plus a verdict on
whether the track had any singing in it at all.

**Kernel:** pick *Python (vocalsplit)* from the kernel menu. Any other kernel will
fail on the first import -- torch and demucs only exist in that environment.

If a track turns out to be instrumental, the notebook says so and **does not write a
vocal file**. That is deliberate: a separator always returns a `vocals` stem, but on
an instrumental track that stem is just bleed -- synth partials and reverb tails --
and saving it produces a confident-looking file full of nothing.

Forward slashes work fine in Windows paths from Python, and avoid backslash-escaping
mistakes.

In [ ]:
SONG    = "C:/Users/ameyn/Music/Kun Faaya Kun Rockstar 320 Kbps.mp3"
OUT_DIR = "C:/Users/ameyn/Music/split"

FORMAT  = "mp3"       # mp3 | wav | flac
BITRATE = 320         # mp3 only
MODEL   = "htdemucs"  # htdemucs_ft is ~4x slower and ~0.5 dB better
SHIFTS  = 0           # 2-5 = test-time augmentation, cleaner but N x slower
DEVICE  = "auto"      # auto | cuda | cpu

## 1. Load the model

First run downloads ~80 MB of weights and caches them. Keep the separator object around -- reloading it per song is pure waste.

In [ ]:
import json, time
from pathlib import Path
import numpy as np

from vocalsplit import DemucsSeparator
from vocalsplit.audio import load_audio, save_audio
from vocalsplit.features import frame_analysis, stem_metrics
from vocalsplit.decide import Thresholds, classify

sep = DemucsSeparator(MODEL, DEVICE, shifts=SHIFTS)
print(f"{MODEL} on {sep.device}  |  sources: {', '.join(sep.sources)}")

## 2. Separate

Unlike the CLI this runs the full separation up front, because the notebook wants the stems and the plot regardless of the verdict.

In [ ]:
t0 = time.perf_counter()
mix, sr = load_audio(SONG, sr=sep.samplerate)
print(f"loaded {mix.shape[1] / sr / 60:.2f} min")

stems = sep.separate(mix, sr)
voc   = stems["vocals"]
inst  = mix[:, : voc.shape[1]] - voc     # exact complement, nothing lost
print(f"separated in {time.perf_counter() - t0:.1f}s")

## 3. Verdict

Four measurements on the vocal stem, each a soft vote. Bleed and singing differ on all four, which is why no single one is trusted alone.

In [ ]:
metrics = stem_metrics(mix, voc, sr)
verdict = classify(metrics, Thresholds())

print(f"{'VERDICT':<18}{verdict['label'].upper()}   p_vocal = {verdict['p_vocal']:.3f}")
print(f"{'':18}{verdict['reasons'][0]}")
print()
for axis, val in verdict["votes"].items():
    print(f"  {axis:<12}{val:5.2f}  " + "#" * int(val * 40))
print()
for k, v in metrics.items():
    print(f"  {k:<18}{v}")

## 4. Where it heard the vocals

The shaded bands are frames counted as vocal-active. On an instrumental track this plot is essentially empty and the red trace sits far below the grey one.

In [ ]:
import matplotlib.pyplot as plt

f = frame_analysis(mix, voc, sr)
t = f["times"] / 60.0

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                       gridspec_kw={"height_ratios": [2, 1]})

ax[0].plot(t, f["mix_db"], lw=0.7, color="#7a7a7a", label="mixture")
ax[0].plot(t, f["vocal_db"], lw=0.7, color="#d1495b", label="vocal stem")
ax[0].fill_between(t, -90, 0, where=f["active"], color="#edae49", alpha=0.30,
                   step="mid", label="counted vocal-active")
ax[0].set_ylim(-90, 0)
ax[0].set_ylabel("dBFS")
ax[0].legend(loc="lower right", fontsize=8, ncol=3)
ax[0].set_title(Path(SONG).stem + "  --  " + verdict["label"]
                + " (p_vocal=" + format(verdict["p_vocal"], ".2f") + ")")

ax[1].plot(t, f["rel_db"], lw=0.7, color="#00798c")
ax[1].axhline(-20, ls="--", lw=1, color="k", label="-20 dB activity gate")
ax[1].set_ylim(-60, 12)
ax[1].set_ylabel("stem - mix (dB)")
ax[1].set_xlabel("minutes")
ax[1].legend(loc="lower right", fontsize=8)

for a in ax:
    a.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Write the files

Both stems get one shared gain if either would clip. A separated stem often peaks above the mixture, and `mixture - vocals` can too, so writing them straight to mp3 would clip silently.

In [ ]:
dest = Path(OUT_DIR) / Path(SONG).stem
dest.mkdir(parents=True, exist_ok=True)
ext = "." + FORMAT

if verdict["label"] == "instrumental":
    print("Instrumental -- no vocal file written.")
    print("Reason: " + verdict["reasons"][0])
    print("To override, set verdict['label'] = 'vocal' and re-run this cell.")
    written = []
else:
    peak = max(float(np.abs(a).max()) for a in (voc, inst))
    gain = min(1.0, 0.999 / peak) if peak > 0 else 1.0
    if gain < 1.0:
        print("peak " + format(20 * np.log10(peak), "+.2f") + " dBFS -> scaling both stems by "
              + format(20 * np.log10(gain), ".2f") + " dB")

    save_audio(dest / ("vocals" + ext),       voc * gain,  sr, bitrate=BITRATE)
    save_audio(dest / ("instrumental" + ext), inst * gain, sr, bitrate=BITRATE)
    written = ["vocals" + ext, "instrumental" + ext]

(dest / "report.json").write_text(json.dumps(
    {"file": SONG, "model": MODEL, **verdict, "metrics": metrics, "written": written}, indent=2))

print()
print("-> " + str(dest))
for w in written + ["report.json"]:
    print("   " + w.ljust(22) + format((dest / w).stat().st_size / 1e6, "7.2f") + " MB")

## 6. Listen

Players are downsampled to 22 kHz mono so the saved notebook stays small. The written files are full quality.

In [ ]:
from IPython.display import Audio, display
import soxr

# Only the first PREVIEW_S seconds, mono at 22 kHz. Audio players embed as base64
# in the .ipynb, so previewing a whole 8-minute track three times over would save
# an ~80 MB notebook. The written mp3s are full length and full quality.
PREVIEW_S = 60

def player(x, label):
    y = x[:, : PREVIEW_S * sr].mean(0)
    print(label + f"  (first {min(PREVIEW_S, x.shape[1] // sr)}s)")
    display(Audio(soxr.resample(y, sr, 22050), rate=22050))

player(mix[:, : voc.shape[1]], "original")
if verdict["label"] != "instrumental":
    player(voc,  "vocals")
    player(inst, "instrumental")

---
### Batch a whole folder

```python
from vocalsplit import Config, analyze, write_outputs
from vocalsplit.audio import find_audio_files

cfg = Config(out_format="mp3", bitrate=320)
for f in find_audio_files("C:/Users/ameyn/Music"):
    write_outputs(analyze(f, cfg, separator=sep), Path(OUT_DIR), cfg)
```

`analyze` runs the cheap probe first -- six 10-second windows spread across the track --
and skips the full separation entirely when those come back confidently instrumental.